
## Scheduling and Monitoring Your Production Pipeline

### Scheduling

- **Schedule**: Runs daily at **6:00 AM**. Verify the timezone is correct.
- **Task chain**: Bronze → Silver → Gold → DQ Checks. Each task starts only after the previous one succeeds.
- **Retries**: Bronze retries **2 times** on failure; Silver, Gold, and DQ retry **1 time**.
- **Catalog parameter**: The pipeline accepts a `catalog` parameter so you can point it at dev, staging, or prod without changing the job definition. In production, pass the production catalog name.

### Monitoring

- **Email alerts**: You get an email when the job fails. Keep the email list current and include a team inbox or on-call rotation.
- **Job run history**: Check the **Runs tab** in the Databricks Jobs UI for success/failure status, duration trends, and retry counts.
- **Task-level visibility**: Because each stage is a separate task, you can see exactly which stage failed.
- **Data quality checks**: The final task validates data quality. Watch DQ pass/fail rates, row counts, and schema drift.
- **System tables**: Use `system.lakeflow.job_runs` for run history and `system.compute.clusters` for cluster utilization. Set up SQL alerts on these tables for proactive monitoring.
- **Recovery on failure**: Use **Repair Run** to re-run from the failed task instead of the whole pipeline.

### Summary Checklist

| Aspect | Status |
|---|---|
| Schedule (6:00 AM) | ✅ |
| Task dependencies | ✅ |
| Retries (Bronze: 2, Others: 1) | ✅ |
| Catalog parameter | ✅ |
| Email notifications | ✅ |

### Job yml

```
resources:
  jobs:
    TPC_H_Job:
      name: TPC-H Job
      email_notifications:
        on_failure:
          - surajitmondal@cyntexa.com
      schedule:
        quartz_cron_expression: 56 0 6 * * ?
        timezone_id: Asia/Calcutta
        pause_status: PAUSED
      tasks:
        - task_key: Bronze_Ingestion
          notebook_task:
            notebook_path: /Workspace/Users/surajitmondal@cyntexa.com/IAP_Exam/Cyntexa_IAP_Exam/Bronze_Ingestion
            base_parameters:
              pipeline_run_id: "{{job.run_id}}"
            source: WORKSPACE
          max_retries: 2
          min_retry_interval_millis: 900000
          disable_auto_optimization: false
        - task_key: Silver_Transform
          depends_on:
            - task_key: Bronze_Ingestion
          notebook_task:
            notebook_path: /Workspace/Users/surajitmondal@cyntexa.com/IAP_Exam/Cyntexa_IAP_Exam/Silver_Transform
            source: WORKSPACE
          max_retries: 1
          min_retry_interval_millis: 900000
          disable_auto_optimization: false
        - task_key: Gold_Model
          depends_on:
            - task_key: Silver_Transform
          notebook_task:
            notebook_path: /Workspace/Users/surajitmondal@cyntexa.com/IAP_Exam/Cyntexa_IAP_Exam/gold_model
            source: WORKSPACE
          max_retries: 1
          min_retry_interval_millis: 900000
          disable_auto_optimization: false
        - task_key: Data_quality_checks
          depends_on:
            - task_key: Gold_Model
          notebook_task:
            notebook_path: /Workspace/Users/surajitmondal@cyntexa.com/IAP_Exam/Cyntexa_IAP_Exam/Data_quality_checks
            source: WORKSPACE
      queue:
        enabled: true
      parameters:
        - name: catalog
          default: tpch_dev
      performance_target: PERFORMANCE_OPTIMIZED
```


![job_screenshot_1792666925171.png](./job_screenshot_1792666925171.png "job_screenshot_1792666925171.png")